In [2]:
# Duplicate of gpt-dev2.15, was the final flushed out version of gpt-dev2 series

# The mathematical trick in self-attention in https://youtu.be/kCc8FmEb1nY?si=MiUse_Fx7msMLc5c&t=2561
# I not going through mathematical trick as I understand it fundamentally and can recall from 8 or 9th standard matrix multiplication class.
# Just know that [B, T, T] @ [B, T, C] --> [B, T, C]

# Removed vocab_size as it is global variable
# Added n_embd instead of 65 dimensions previously (vocab_size in our example is 65 and we had vocab_size x vocab_size embedding)
# n_embd = 32 is a good number. In make more I think we had 2 first and then we increased it to 10
# if we think, n_embd is basically on z-axis in my mental world. It is C in BTC

# Adding n_embd will give us a way to add a level of INDIRECTION. 

# Introducing HEAD (which is important)
# previously we had `logits = self.token_embedding_table(idx)` in `def forward`
# now, as we will introduce more layers, `self.token_embedding_table(idx)` will NOT give us logits.
# Instead it will give us tok_emb
# So, in order for us to go from tok_emb to logits, we will need linear layer.
# so that linear layer is call head. Will add self.lm_head to our bigram model


In [5]:
# bigram.py final
# Putting all things together from gpt-dev2 series through gpt-dev2.14
# Commented "# Added" for lines added just for this code
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 32 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?
max_iters = 3000
eval_interval = 300
learning_rate = 1e-2
device = 'cuda' if torch.cuda.is_available() else 'cpu' # Added
eval_iters = 200
n_embd = 32
# ------------

torch.manual_seed(1337)

# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad() # will not call .backward on these tensors. More memory efficient.
def estimate_loss(): # Added
    out = {}
    model.eval() # this doesn't make any sense, since there are no drop out or batch normalization layers.
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

# super simple bigram model
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        # Remember, we are mapping from smaller C = 32 to larger vocab_size = 65
        logits = self.lm_head(tok_emb) # (B, T, vocab_size)
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

model = BigramLanguageModel()
m = model.to(device)

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0:
        losses = estimate_loss() # Added
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device) # Added
print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))

step 0: train loss 4.3886, val loss 4.3734
step 300: train loss 2.5267, val loss 2.5399
step 600: train loss 2.4998, val loss 2.5315
step 900: train loss 2.4903, val loss 2.5085
step 1200: train loss 2.4967, val loss 2.5128
step 1500: train loss 2.4809, val loss 2.5020
step 1800: train loss 2.4858, val loss 2.5149
step 2100: train loss 2.4865, val loss 2.5000
step 2400: train loss 2.4882, val loss 2.5127
step 2700: train loss 2.5006, val loss 2.5117

A:
Wit min fush IUnonthefove thacke nd al INepe unco mid at IIfuries rye.

Cis vedis to ctha dry chit MEOffis cebans ur rf y, ieve dse lathad f yive;
Tharard il tos oulop ope martthor'ds id t RTEN:
Tofavalx el r y, lat condin'macharus pes son:
Cose;
THesonthonde'
THAnd ithi:
LANalistarmuctimis to blllondedala!

NG espro. hin g
Fo d cheo.
Asell sik thangs,-h
TOrthathe mis trt HAnju
Clyofflloutis mouroff veal'rali t aliole O:
Sathanous wistrid dsthanullt, louqulthiseswen t hyoutelfietrduee ilais's


In [3]:
# The mathmatical trick in self-attention
# https://youtu.be/kCc8FmEb1nY?si=MiUse_Fx7msMLc5c&t=2561

# consider the following toy example:

torch.manual_seed(1337)
B,T,C = 4,8,2 # batch, time, channels
x = torch.randn(B,T,C)
x.shape

# We want x[b,t] = mean_{i<=t} x[b,i]
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1] # (t,C)
        xbow[b,t] = torch.mean(xprev, 0)

In [4]:
torch.tril(torch.ones(3, 3))

tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])